In [35]:
import gc
import os
import sys
import math

import numpy as np
import optuna
import pandas as pd
import torch
from sklearn.model_selection import KFold

current_dir = os.getcwd()
parent_dir = os.path.abspath(os.path.join(current_dir, ".."))
sys.path.append(parent_dir)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [16]:
%load_ext autoreload
%autoreload 2
from drGAT import drGAT

from drGAT.load_data import load_data
from drGAT.sampler import RandomSampler

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [17]:
drugAct, pos_num, null_mask, S_d, S_c, S_g, A_cg, A_dg, _, _, _ = load_data('nci')

load nci
unique drugs: 177
unique genes: 251
DTI unique genes:  251
Top 90% variable genes:  2383
Total:  2582
Final gene exp shape: (60, 2582)
Final drug Act shape: (1005, 60)


100%|██████████| 25/25 [00:02<00:00,  9.36it/s]

Done!


In [5]:
PATH = "../nci_data/"

In [18]:
method = "GATv2"
study = optuna.create_study(
    storage=f"sqlite:///{method}_small.sqlite3",
    study_name=method,
    load_if_exists=True,
)

[I 2025-04-15 16:34:11,655] Using an existing study with name 'GATv2' instead of creating a new one.


In [36]:
def auto_convert_params(params, nan_replace=None):
    """Convert parameter types automatically

    Args:
        params (dict): Parameter dictionary before conversion
        nan_replace: Replacement value for NaN (default None)

    Returns:
        dict: Parameter dictionary after type conversion
    """
    converted = {}
    for k, v in params.items():
        if isinstance(v, float) and math.isnan(v):
            converted[k] = nan_replace
        elif isinstance(v, float) and v.is_integer():
            converted[k] = int(v)
        else:
            converted[k] = v
    return converted

In [39]:
params = study.best_trials[0].params
params = auto_convert_params(params, nan_replace=0)
params.update(
    {
        "n_drug": S_d.shape[0],
        "n_cell": S_c.shape[0],
        "n_gene": S_g.shape[0],
        "epochs": 100,
        "gnn_layer": method,
    }
)
params

{'dropout1': 0.2,
 'dropout2': 0.1,
 'hidden1': 305,
 'hidden2': 85,
 'hidden3': 79,
 'heads': 4,
 'activation': 'gelu',
 'optimizer': 'Adam',
 'lr': 0.00047762282267892084,
 'weight_decay': 6.656420345368285e-05,
 'scheduler': None,
 'n_drug': 1005,
 'n_cell': 60,
 'n_gene': 2582,
 'epochs': 100,
 'gnn_layer': 'GATv2'}

In [40]:
k = 5
kfold = KFold(n_splits=k, shuffle=True, random_state=42)

true_datas = pd.DataFrame()
predict_datas = pd.DataFrame()

for seed, (train_index, test_index) in enumerate(kfold.split(np.arange(pos_num))):
    sampler = RandomSampler(
        drugAct.T,
        train_index,
        test_index,
        null_mask.T,
        S_d,
        S_c,
        S_g,
        A_cg,
        A_dg,
        PATH,
        seed
    )
    (
            _,
            train_attention,
            val_attention,_,_,_,_,_,_,
    ) = drGAT.train(
        sampler, params=params, device=device, verbose=False
    )
    break

Using device: cpu


/Users/inouey2/miniconda3/envs/torch/lib/python3.10/site-packages/torch/amp/grad_scaler.py:132: UserWarning: torch.cuda.amp.GradScaler is enabled, but CUDA is not available.  Disabling.
  warnings.warn(


KeyboardInterrupt: 